<a href="https://colab.research.google.com/github/EugeneOny1/-civil-infrastructure-monitoring---CS-PROJECT-II/blob/feature-branch/03_annotation_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03 — Annotation Validation and Dataset Preparation

This notebook:
- validates RDD2022 YOLO annotations;
- validates manually created Pascal VOC annotations for SDNET2018 and Kenyan_Data;
- maps source labels to the three project classes;
- creates a normalized annotation manifest;
- creates train/validation/test splits;
- keeps the original Raw Data untouched.

**Important:** Run Notebook 02 first and confirm the observed RDD2022 class IDs.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import random
import shutil
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from PIL import Image

PROJECT_DIR = Path("/content/drive/MyDrive/CS PROJECT II")
DATA_DIR = PROJECT_DIR / "Data"

RAW_DIR = DATA_DIR / "Raw Data"
SELECTED_DIR = DATA_DIR / "Selected Data"
ANNOTATED_DIR = DATA_DIR / "Annotated Data"
PROCESSED_DIR = DATA_DIR / "Processed Data"

RDD_DIR = RAW_DIR / "RDD2022"
SDNET_ANNOTATED = ANNOTATED_DIR / "SDNET2018"
KENYAN_ANNOTATED = ANNOTATED_DIR / "Kenyan_Data"

PROCESSED_ANNOTATIONS = PROCESSED_DIR / "annotations"
PROCESSED_ANNOTATIONS.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = {
    1: "Crack",
    2: "Pothole",
    3: "Surface deterioration",
}

# IMPORTANT CONFIRMATION:
# Notebook 02 must show the actual RDD2022 IDs. The standard RDD2022 mapping
# expected for this project is 0=D00, 1=D10, 2=D20, 3=D40.
# If Notebook 02 shows a different numbering, change this dictionary before continuing.
RDD_CLASS_ID_TO_CODE = {
    0: "D00",
    1: "D10",
    2: "D20",
    3: "D40",
}

RDD_CODE_TO_TARGET = {
    "D00": "Crack",
    "D10": "Crack",
    "D20": "Crack",
    "D40": "Pothole",
}

SOURCE_PRIORITY = {
    "RDD2022": 1,
    "SDNET2018": 2,
    "Kenyan_Data": 3,
}

print(TARGET_CLASSES)

## 1. Annotation Parsers

The project currently has two annotation situations:

- RDD2022: YOLO `.txt`
- Manually annotated SDNET2018/Kenyan_Data: Pascal VOC `.xml`

Do not force one format into another until the information has been validated.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG", ".PNG", ".WEBP"}

def image_for_annotation(annotation_path):
    annotation_path = Path(annotation_path)
    candidates = []
    for ext in IMAGE_EXTENSIONS:
        candidates.append(annotation_path.with_suffix(ext))

    for candidate in candidates:
        if candidate.exists():
            return candidate

    # Fallback: search in the same directory for the same stem.
    matches = list(annotation_path.parent.glob(annotation_path.stem + ".*"))
    for match in matches:
        if match.suffix in IMAGE_EXTENSIONS:
            return match

    return None

def parse_yolo_file(txt_path):
    objects = []
    errors = []

    for line_no, line in enumerate(
        open(txt_path, "r", encoding="utf-8", errors="ignore"),
        start=1
    ):
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) != 5:
            errors.append(f"Line {line_no}: expected 5 fields, got {len(parts)}")
            continue

        try:
            class_id = int(parts[0])
            xc, yc, w, h = map(float, parts[1:])
        except ValueError:
            errors.append(f"Line {line_no}: non-numeric value")
            continue

        objects.append({
            "class_id": class_id,
            "xc": xc,
            "yc": yc,
            "width": w,
            "height": h,
        })

    return objects, errors

def parse_voc_file(xml_path):
    root = ET.parse(xml_path).getroot()

    size = root.find("size")
    if size is None:
        raise ValueError("VOC file has no <size> element")

    width = int(size.findtext("width", "0"))
    height = int(size.findtext("height", "0"))

    objects = []
    for obj in root.findall("object"):
        name = obj.findtext("name", "").strip()

        bbox = obj.find("bndbox")
        if bbox is None:
            continue

        xmin = float(bbox.findtext("xmin", "0"))
        ymin = float(bbox.findtext("ymin", "0"))
        xmax = float(bbox.findtext("xmax", "0"))
        ymax = float(bbox.findtext("ymax", "0"))

        objects.append({
            "source_class": name,
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
            "width": width,
            "height": height,
        })

    return width, height, objects

## 2. Validate RDD2022 YOLO Annotations

In [ ]:
def yolo_to_normalized_box(xc, yc, w, h):
    xmin = xc - w / 2
    ymin = yc - h / 2
    xmax = xc + w / 2
    ymax = yc + h / 2
    return xmin, ymin, xmax, ymax

rdd_records = []
rdd_errors = []
rdd_class_counts = Counter()

rdd_txt_files = list(RDD_DIR.rglob("*.txt"))

for txt_path in rdd_txt_files:
    image_path = image_for_annotation(txt_path)

    if image_path is None:
        rdd_errors.append({
            "file": str(txt_path),
            "error": "No matching image found"
        })
        continue

    try:
        with Image.open(image_path) as img:
            img.verify()
    except Exception as exc:
        rdd_errors.append({
            "file": str(image_path),
            "error": f"Unreadable image: {exc}"
        })
        continue

    objects, parse_errors = parse_yolo_file(txt_path)

    for err in parse_errors:
        rdd_errors.append({"file": str(txt_path), "error": err})

    normalized_objects = []

    for obj in objects:
        class_id = obj["class_id"]

        if class_id not in RDD_CLASS_ID_TO_CODE:
            rdd_errors.append({
                "file": str(txt_path),
                "error": f"Unknown RDD class ID: {class_id}"
            })
            continue

        if not (0 <= obj["xc"] <= 1 and 0 <= obj["yc"] <= 1 and
                0 < obj["width"] <= 1 and 0 < obj["height"] <= 1):
            rdd_errors.append({
                "file": str(txt_path),
                "error": f"Invalid normalized coordinates: {obj}"
            })
            continue

        xmin, ymin, xmax, ymax = yolo_to_normalized_box(
            obj["xc"], obj["yc"], obj["width"], obj["height"]
        )

        if not (0 <= xmin < xmax <= 1 and 0 <= ymin < ymax <= 1):
            rdd_errors.append({
                "file": str(txt_path),
                "error": f"Bounding box outside [0,1]: {obj}"
            })
            continue

        source_code = RDD_CLASS_ID_TO_CODE[class_id]
        target_class = RDD_CODE_TO_TARGET[source_code]
        rdd_class_counts[target_class] += 1

        normalized_objects.append({
            "class_name": target_class,
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
            "source_class": source_code,
        })

    if normalized_objects:
        rdd_records.append({
            "dataset": "RDD2022",
            "image_path": str(image_path),
            "annotation_path": str(txt_path),
            "annotation_format": "YOLO",
            "objects": normalized_objects,
        })

print("Valid RDD records:", len(rdd_records))
print("RDD errors:", len(rdd_errors))
print("RDD target-class counts:", dict(rdd_class_counts))

## 3. Validate Pascal VOC Annotations

This section expects manually annotated images exported as Pascal VOC.

Expected folders:

```text
Annotated Data/
├── SDNET2018/
│   ├── images/
│   └── annotations/
└── Kenyan_Data/
    ├── images/
    └── annotations/
```

If your DigitalSreeni export uses different folder names, change the paths below only.

In [ ]:
VOC_CLASS_MAP = {
    "crack": "Crack",
    "cracks": "Crack",
    "pothole": "Pothole",
    "potholes": "Pothole",
    "surface deterioration": "Surface deterioration",
    "surface_deterioration": "Surface deterioration",
    "surface deterioration ": "Surface deterioration",
}

def find_voc_image(xml_path):
    return image_for_annotation(xml_path)

def validate_voc_dataset(dataset_name, dataset_root):
    records = []
    errors = []

    xml_files = list(Path(dataset_root).rglob("*.xml"))

    for xml_path in xml_files:
        image_path = find_voc_image(xml_path)

        if image_path is None:
            errors.append({"file": str(xml_path), "error": "Matching image not found"})
            continue

        try:
            width, height, objects = parse_voc_file(xml_path)
        except Exception as exc:
            errors.append({"file": str(xml_path), "error": str(exc)})
            continue

        if width <= 0 or height <= 0:
            errors.append({"file": str(xml_path), "error": "Invalid image dimensions"})
            continue

        normalized_objects = []

        for obj in objects:
            raw_name = obj["source_class"].strip().lower()

            if raw_name not in VOC_CLASS_MAP:
                errors.append({
                    "file": str(xml_path),
                    "error": f"Unknown class name: {obj['source_class']}"
                })
                continue

            xmin = obj["xmin"] / width
            ymin = obj["ymin"] / height
            xmax = obj["xmax"] / width
            ymax = obj["ymax"] / height

            if not (0 <= xmin < xmax <= 1 and 0 <= ymin < ymax <= 1):
                errors.append({
                    "file": str(xml_path),
                    "error": f"Invalid bounding box: {obj}"
                })
                continue

            target_class = VOC_CLASS_MAP[raw_name]

            normalized_objects.append({
                "class_name": target_class,
                "xmin": xmin,
                "ymin": ymin,
                "xmax": xmax,
                "ymax": ymax,
                "source_class": obj["source_class"],
            })

        if normalized_objects:
            records.append({
                "dataset": dataset_name,
                "image_path": str(image_path),
                "annotation_path": str(xml_path),
                "annotation_format": "PascalVOC",
                "objects": normalized_objects,
            })

    return records, errors

sdnet_records, sdnet_errors = validate_voc_dataset("SDNET2018", SDNET_ANNOTATED)
kenyan_records, kenyan_errors = validate_voc_dataset("Kenyan_Data", KENYAN_ANNOTATED)

print("SDNET valid records:", len(sdnet_records), "errors:", len(sdnet_errors))
print("Kenyan valid records:", len(kenyan_records), "errors:", len(kenyan_errors))

## 4. Important SDNET2018 Note

Do **not** automatically assign a full-image bounding box to every SDNET2018 crack patch.

For this project, selected SDNET images that are being used for object detection should have the visible defect manually bounded where practical. A full-image box would teach the detector that the entire patch is the defect and would provide weak localization evidence.

If the selected SDNET data has not yet been manually annotated, stop here and complete the required annotation before using those images in training.

In [ ]:
all_records = rdd_records + sdnet_records + kenyan_records

records_df = pd.DataFrame([
    {
        "dataset": r["dataset"],
        "image_path": r["image_path"],
        "annotation_path": r["annotation_path"],
        "annotation_format": r["annotation_format"],
        "object_count": len(r["objects"]),
        "classes": sorted(set(o["class_name"] for o in r["objects"]))
    }
    for r in all_records
])

display(records_df.groupby(["dataset", "annotation_format"]).size().reset_index(name="records"))

all_class_counts = Counter()
for r in all_records:
    for obj in r["objects"]:
        all_class_counts[obj["class_name"]] += 1

print("Unified object counts:")
print(dict(all_class_counts))

## 5. Create a Normalized Manifest

The manifest is the hand-off between annotation validation and TFRecord generation.

It contains:
- image path
- annotation source
- normalized bounding boxes
- target class names

The Raw Data is not modified.

In [ ]:
MANIFEST_PATH = PROCESSED_ANNOTATIONS / "validated_manifest.jsonl"

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    for record in all_records:
        f.write(json.dumps(record) + "\n")

print("Manifest written to:", MANIFEST_PATH)
print("Records:", len(all_records))

## 6. Train / Validation / Test Split

The split is performed at the image level so an image cannot appear in more than one split.

For Kenyan_Data, retain a local test portion so the final evaluation can report performance on locally sourced images.

If you have multiple photographs of the same physical location/site, those images should ideally be grouped before splitting to prevent location leakage.

In [ ]:
from sklearn.model_selection import train_test_split

random_seed = 42

# Load records from the manifest.
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

# Separate Kenyan data so some local examples remain held out.
kenyan = [r for r in records if r["dataset"] == "Kenyan_Data"]
non_kenyan = [r for r in records if r["dataset"] != "Kenyan_Data"]

def split_records(items, test_size, val_size_from_remaining, seed=42):
    if len(items) < 3:
        return items, [], []

    train_val, test = train_test_split(
        items, test_size=test_size, random_state=seed
    )

    val_fraction = val_size_from_remaining
    if len(train_val) < 2:
        return train_val, [], test

    train, val = train_test_split(
        train_val, test_size=val_fraction, random_state=seed
    )
    return train, val, test

train_non, val_non, test_non = split_records(
    non_kenyan,
    test_size=0.15,
    val_size_from_remaining=0.1765,  # approximately 15% of total
    seed=random_seed,
)

# Kenyan: 70/15/15 where possible, preserving a local test set.
train_ke, val_ke, test_ke = split_records(
    kenyan,
    test_size=0.20,
    val_size_from_remaining=0.1875,
    seed=random_seed,
)

train_records = train_non + train_ke
val_records = val_non + val_ke
test_records = test_non + test_ke

random.Random(random_seed).shuffle(train_records)
random.Random(random_seed).shuffle(val_records)
random.Random(random_seed).shuffle(test_records)

print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))

In [ ]:
SPLIT_DIR = PROCESSED_ANNOTATIONS / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, split_records_list in {
    "train": train_records,
    "validation": val_records,
    "test": test_records,
}.items():
    path = SPLIT_DIR / f"{split_name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for record in split_records_list:
            f.write(json.dumps(record) + "\n")
    print(split_name, "->", path)

In [ ]:
def split_class_counts(split_records):
    counts = Counter()
    for r in split_records:
        for obj in r["objects"]:
            counts[obj["class_name"]] += 1
    return dict(counts)

print("Train classes:", split_class_counts(train_records))
print("Validation classes:", split_class_counts(val_records))
print("Test classes:", split_class_counts(test_records))

print("\nCHECK: Every target class should ideally occur in all three splits.")
for split_name, split_records_list in [
    ("train", train_records),
    ("validation", val_records),
    ("test", test_records),
]:
    present = set(split_class_counts(split_records_list))
    missing = set(TARGET_CLASSES.values()) - present
    print(split_name, "missing:", missing)

## 7. Final Annotation QC

Review the error lists before training.

If errors are substantial, fix the source annotations and rerun this notebook. Do not silently discard large portions of the dataset.

In [ ]:
print("RDD errors:", len(rdd_errors))
print("SDNET errors:", len(sdnet_errors))
print("Kenyan errors:", len(kenyan_errors))

for title, errors in [
    ("RDD2022", rdd_errors),
    ("SDNET2018", sdnet_errors),
    ("Kenyan_Data", kenyan_errors),
]:
    if errors:
        print(f"\n{title} error examples:")
        for error in errors[:10]:
            print(error)